# Agent Identity (Governance)

In [1]:
# @title Installs
############################################

# Google Cloud Vertex AI (Agent Engine) & Google ADK installieren
# Feste Versionen statt --upgrade: Der Deploy pinnt dieselben Versionen, damit das Pickle in der Cloud passt
%pip install --quiet "google-cloud-aiplatform[agent_engines,adk]==2.6.0" "google-adk==2.12.0" "google-genai==2.33.0"

# Versionen kurz prüfen
%pip show google-adk google-cloud-aiplatform

Note: you may need to restart the kernel to use updated packages.
Name: google-adk
Version: 2.12.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: aiosqlite, authlib, click, cryptography, fastapi, google-auth, google-genai, graphviz, httpx, jsonschema, opentelemetry-api, opentelemetry-sdk, packaging, pydantic, python-dotenv, python-multipart, pyyaml, requests, starlette, tenacity, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
---
Name: google-cloud-aiplatform
Version: 2.6.0
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: certifi, docstring_parser, google-api-core, google-auth, google-cloud-bigquery, google-cl

In [2]:
# @title Imports
############################################

import logging
import os
import warnings
from importlib.metadata import version

import vertexai
from google.adk.agents import LlmAgent
from google.adk.integrations.agent_registry import AgentRegistry
from google.genai.errors import ClientError
from vertexai import types
from vertexai.agent_engines import AdkApp

warnings.filterwarnings('ignore')

print("✅ Alle Imports erfolgreich geladen!")

/opt/anaconda3/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


✅ Alle Imports erfolgreich geladen!


In [3]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"  # Agent Engine Region
STAGING_BUCKET = "gs://deltorobarba-agent-staging"
GCS_DIR = "agent_governance"  # eigener Ordner im Bucket: agents.ipynb benutzt 'agent_engine/'
MODEL = "gemini-3.8-flash"

# Nach dem ersten Deploy hier eintragen, damit ein zweiter Lauf keine weitere Engine anlegt
RESOURCE_NAME = ""

# These tell the underlying google-genai client to use Vertex AI (Gemini 3.8 Flash uses 'global')
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Initialize Vertex AI SDK & Client (v1beta1: nötig für identity_type und agent_gateway_config)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION, http_options=dict(api_version="v1beta1"))

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

✅ Erfolgreich über Application Default Credentials (ADC) verbunden.


##### <font color="blue">*Agent Identity*

Ein einzelner Agent ohne Tools wird auf Agent Runtime mit eigener **Agent Identity** deployt und direkt abgefragt.

```mermaid
%%{init: {'themeVariables': {'fontSize': '13px'}, 'flowchart': {'nodeSpacing': 20, 'rankSpacing': 28, 'padding': 6}}}%%
flowchart LR
    U(["Du<br/>Access-Token"]) -->|"query"| A["agent_identity<br/>Agent Identity (SPIFFE)"]
    A -->|"eigene Identität"| G[("Gemini")]
    A -.- R["Agent Registry"]
    GW["Agent Gateway<br/>optional"] -.- A
```

* **Was:** Mit `identity_type=AGENT_IDENTITY` bekommt die Engine eine eigene, von Google verwaltete Identität statt des Reasoning-Engine-Service-Agents: eine SPIFFE-ID wie `agents.global.proj-<Projektnummer>.system.id.goog/resources/aiplatform/projects/<Nr>/locations/<Region>/reasoningEngines/<ID>`. Gehört das Projekt zu einer Organisation, steht dort `org-<Org-ID>` statt `proj-<Projektnummer>`.
* **Kein Schlüssel:** Es gibt keinen Service Account und keinen Key. Das Token ist an ein Zertifikat der Runtime gebunden und gilt nur dort.
* **IAM:** Rollen gehen direkt an den Principal `principal://<SPIFFE-ID>`. Beim Erstellen gibt Google allen Agent Identities des Projekts `roles/aiplatform.agentDefaultAccess` (Modell, Sessions, Memory, Logging); für diesen Agent reicht das.
* **Nur beim `create`:** Die Identität lässt sich nicht per `update` einschalten. Wird die Engine gelöscht und neu erstellt, entsteht eine neue Identität, und alte Rollen gelten für sie nicht.
* **Voraussetzung für Agent Gateway** (unten).

In [4]:
# @title Define Agent
############################################

# Ein einzelner Agent ohne Tools: braucht außer den Standardrechten seiner Agent Identity nichts
root_agent = LlmAgent(
    name="agent_identity",
    model=MODEL,
    description="Minimal agent to test Agent Identity, Agent Registry and Agent Gateway.",
    instruction="You are a concise assistant. Answer in at most three sentences.",
)

print(f"✅ Agent '{root_agent.name}' definiert.")

✅ Agent 'agent_identity' definiert.


In [5]:
# @title Local Test (Before Deploying to Agent Engine)
############################################
# Läuft mit DEINEN Credentials: prüft den Agent, noch nicht seine Identität

local_app = AdkApp(agent=root_agent)


async def run_local_test():
    user_id = "deltorobarba"
    session = await local_app.async_create_session(user_id=user_id)
    print(f"🧪 Local Test Session ID: {session['id']}")

    async for event in local_app.async_stream_query(
        user_id=user_id,
        session_id=session["id"],
        message="Explain in two sentences what a SPIFFE ID is.",
    ):
        for part in event.get("content", {}).get("parts", []):
            if "text" in part:
                print(f"\n--- Local Output ({event.get('author')}) ---\n{part['text']}")

await run_local_test()

🧪 Local Test Session ID: 1f6ba959-f831-422c-941f-a2b1e133955b

--- Local Output (agent_identity) ---
A SPIFFE ID is a standardized uniform resource identifier (URI) used to uniquely and securely identify software workloads across dynamic, heterogeneous environments. Formatted as `spiffe://<trust-domain>/<workload-path>`, it is embedded into cryptographically verifiable documents like X.509 certificates to establish workload identity and enable secure mutual authentication.


In [ ]:
# @title Deploy Agent with Agent Identity on Agent Engine (⚠️ --> once only!)
############################################

"""
Agent Identity: https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/agent-identity
identity_type=AGENT_IDENTITY geht nur beim create; danach RESOURCE_NAME oben eintragen.
"""

# Fortschritt anzeigen: das SDK loggt LRO und Resource-Namen auf INFO, was Jupyter sonst verschluckt
_log = logging.getLogger("vertexai_genai.agentengines")
_log.setLevel(logging.INFO)
_log.propagate = False
if not _log.handlers:
    _log.addHandler(logging.StreamHandler())

DEPLOY_CONFIG = {
    "display_name": "Agent Identity",
    "description": "Single agent with Agent Identity",
    "staging_bucket": STAGING_BUCKET,
    "gcs_dir_name": f"{GCS_DIR}/agent_identity",  # eigener Ordner: das Pickle von agents.ipynb bleibt unberührt
    # Same versions as locally: the agent is pickled here and unpickled in the cloud
    "requirements": [
        f"google-cloud-aiplatform[agent_engines,adk]=={version('google-cloud-aiplatform')}",
        f"google-adk=={version('google-adk')}",
        f"google-genai=={version('google-genai')}",
        f"cloudpickle=={version('cloudpickle')}",
        f"pydantic=={version('pydantic')}",
    ],
    "env_vars": {"GOOGLE_CLOUD_LOCATION": "global"},  # Gemini 3.8 Flash läuft auf 'global'
}

if RESOURCE_NAME:
    print("Engine existiert schon:", RESOURCE_NAME)
else:
    remote_agent = client.agent_engines.create(
        agent=AdkApp(agent=root_agent),
        config={**DEPLOY_CONFIG, "identity_type": types.IdentityType.AGENT_IDENTITY},
    )
    RESOURCE_NAME = remote_agent.api_resource.name
    print(f"\n✅ Engine angelegt: {RESOURCE_NAME}")
    print("   👉 oben in 'Environmental Variables & GCP Setup' als RESOURCE_NAME eintragen")

AGENT_IDENTITY = client.agent_engines.get(name=RESOURCE_NAME).api_resource.spec.effective_identity
AGENT_PRINCIPAL = f"principal://{AGENT_IDENTITY}"
print("SPIFFE-ID: ", AGENT_IDENTITY)
print("IAM-Member:", AGENT_PRINCIPAL)

In [ ]:
# @title (Optional) IAM-Rollen für die Agent Identity
############################################
# Für diesen Agent nicht nötig (Standardrechte, siehe oben). Die Google-Doku empfiehlt für Agents diese
# Rollen; Agents mit Tools brauchen zusätzlich die Rollen ihrer Zielsysteme (z. B. discoveryengine.viewer).

AGENT_ROLES = [
    "roles/aiplatform.expressUser",             # Inferenz, Sessions, Memory
    "roles/serviceusage.serviceUsageConsumer",  # Kontingent des Projekts nutzen
    "roles/browser",                            # Grundfunktionen von Google Cloud
]
for role in AGENT_ROLES:
    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="{role}" --condition=None --quiet > /dev/null && echo "✅ {role}"
print("IAM-Propagation dauert einige Minuten.")

In [ ]:
# @title Test Agent ====> Single Turn Query
############################################
# Du rufst mit deinem Access-Token auf; der Agent legt die Session an und ruft Gemini mit SEINER Identität

remote_app = client.agent_engines.get(name=RESOURCE_NAME)


async def run_remote_test():
    user_id = "deltorobarba"
    print(f"Connecting to: {RESOURCE_NAME}...")
    try:
        session = await remote_app.async_create_session(user_id=user_id)
        print(f"✅ Managed Session ID: {session['id']}")

        async for event in remote_app.async_stream_query(
            user_id=user_id,
            session_id=session["id"],
            message="Explain in two sentences what a SPIFFE ID is.",
        ):
            if event.get("error_message"):
                print(f"❌ [{event.get('author')}] {event.get('error_code')}: {event['error_message']}")
            for part in event.get("content", {}).get("parts", []):
                if "text" in part:
                    print(f"\n--- Agent Output ({event.get('author')}) ---\n{part['text']}")
    except ClientError as e:
        print("❌", e.message)
        print("\n-> siehe Troubleshooting am Ende")

await run_remote_test()

##### <font color="blue">*Agent Registry*

Jede Engine auf Agent Runtime steht automatisch in der Agent Registry (Console: Agent Platform › Govern › Agent Registry), mit Framework, Protokollen und **Laufzeit-Identität**; ein eigener Registrierungsschritt ist nicht nötig. So sieht man an einer Stelle, welcher Agent als wer läuft: die Pipeline aus `agents.ipynb` mit `sa://service-…` (Reasoning-Engine-Service-Agent), dieser Agent mit `principal://agents.global.…`.

In [ ]:
# @title Agent Registry: Eintrag und Laufzeit-Identität
############################################
# Der Eintrag erscheint wenige Minuten nach dem Deploy

agent_registry = AgentRegistry(project_id=PROJECT_ID, location=LOCATION)
RUNTIME_KEY = "agentregistry.googleapis.com/system/RuntimeReference"
IDENTITY_KEY = "agentregistry.googleapis.com/system/RuntimeIdentity"

registered = agent_registry.list_agents().get("agents", [])
print("Alle registrierten Agents mit Laufzeit-Identität:")
for a in registered:
    principal = a.get("attributes", {}).get(IDENTITY_KEY, {}).get("principal", "–")
    print(f"  {a.get('displayName', '?'):22} {principal}")

engine_id = RESOURCE_NAME.split("/")[-1]
entry = next((a for a in registered
              if a.get("attributes", {}).get(RUNTIME_KEY, {}).get("uri", "").endswith(engine_id)), None)
if entry is None:
    print("\nDieser Agent ist noch nicht in der Registry – in ein paar Minuten erneut ausführen.")
else:
    principal = entry["attributes"].get(IDENTITY_KEY, {}).get("principal")
    print("\nDieser Agent:")
    print("  Registry-Eintrag:  ", entry["name"])
    print("  Protokolle:        ", [p.get("type") for p in entry.get("protocols", [])])
    print("  Laufzeit-Identität:", principal)
    print("  = SPIFFE-ID der Engine:", principal == AGENT_PRINCIPAL)

##### <font color="blue">*Agent Gateway*

Ein Agent Gateway ist der zentrale Ein- und Ausgang für Agent-Verkehr: Zugriff nach IAM- und Policy-Regeln, Logging an einer Stelle.

| Richtung | `governedAccessPath` | Bindung an der Engine | Wirkung |
|---|---|---|---|
| Eingang | `CLIENT_TO_AGENT` | `client_to_agent_config` | Alle Aufrufe des Agents laufen über das Gateway |
| Ausgang | `AGENT_TO_ANYWHERE` | `agent_to_anywhere_config` | Alle ausgehenden Aufrufe des Agents laufen über das Gateway |

* **Voraussetzungen:** Die Engine hat Agent Identity, und das Gateway existiert schon. Angelegt wird es in der Console (Agent Platform › Govern › Agent Gateway), mit Terraform (`google_network_services_agent_gateway`) oder mit `gcloud network-services agent-gateways import`.
* **Binden** ist ein `update`, das den Code (`agent=`) mitschicken muss; die Identität bleibt dabei erhalten.
* **Ausgang fehlt hier bewusst:** Das Gateway entschlüsselt und prüft ausgehenden TLS-Verkehr, also muss das Container-Image seiner Root-CA vertrauen. Das geht nur mit einem eigenen Dockerfile (container-basierter Deploy, z. B. `agents-cli scaffold enhance . --agent-gateway`), nicht mit dem Pickle-Deploy dieses Notebooks.

In [ ]:
# @title Agent Gateway: Agent am Eingang binden (optional)
############################################

!gcloud network-services agent-gateways list --location={LOCATION} --project={PROJECT_ID}

# Vollständiger Name eines Gateways mit governedAccessPath=CLIENT_TO_AGENT, z. B.
# "projects/deltorobarba/locations/us-central1/agentGateways/<name>"
AGENT_GATEWAY_INGRESS = ""

if not AGENT_GATEWAY_INGRESS:
    print("Kein Gateway eingetragen: Binden übersprungen.")
else:
    gateway_config = {"client_to_agent_config": {"agent_gateway": AGENT_GATEWAY_INGRESS}}
    client.agent_engines.update(
        name=RESOURCE_NAME,
        agent=AdkApp(agent=root_agent),
        config={**DEPLOY_CONFIG, "agent_gateway_config": gateway_config},
    )
    spec = client.agent_engines.get(name=RESOURCE_NAME).api_resource.spec
    print("✅ gebunden:", spec.deployment_spec.agent_gateway_config)
    print("   Identität unverändert:", spec.effective_identity == AGENT_IDENTITY)
    print("Danach 'Test Agent' erneut ausführen: der Aufruf läuft jetzt über das Gateway.")

In [ ]:
# @title (Optional) Aufräumen
############################################
# Löscht nur die Ressourcen dieses Notebooks; agents.ipynb bleibt unberührt.
# IAM-Bindings auf gelöschte principal://-Identitäten bleiben sonst stehen.

# client.agent_engines.delete(name=RESOURCE_NAME, force=True)
# for role in AGENT_ROLES:
#     !gcloud projects remove-iam-policy-binding {PROJECT_ID} --member="{AGENT_PRINCIPAL}" --role="{role}" --condition=None --quiet > /dev/null
# !gcloud storage rm -r {STAGING_BUCKET}/{GCS_DIR}

##### <font color="blue">*Troubleshooting*

| Symptom | Ursache / Fix |
|---|---|
| `identity_type` unbekannt oder ignoriert | Client ohne `http_options=dict(api_version="v1beta1")` (Setup) |
| 403 `aiplatform.sessions.create` oder `aiplatform.endpoints.predict` im Test, auch lange nach dem Deploy | Die Identität hat keine wirksamen Rechte, obwohl Google `agentDefaultAccess` vergibt. Beobachtet, als das Projekt einer Organisation zugeordnet war (Trust-Domain `org-<Org-ID>`): die Ursache liegt dann oberhalb des Projekts. Prüfen in *IAM & Admin › Policy Troubleshooter* mit Principal = `AGENT_PRINCIPAL`, Ressource = `//aiplatform.googleapis.com/` + `RESOURCE_NAME`, Berechtigung = `aiplatform.sessions.create`; die Antwort zeigt Allow-, Deny- und PAB-Entscheidung getrennt |
| 401 `Context-Aware Access requirements are not met` | Token der Agent Identity außerhalb der Runtime benutzt – gewollt, das Token ist an die Runtime gebunden |
| Gateway-Bindung schlägt fehl | Engine ohne Agent Identity (nur beim `create` wählbar) oder Gateway mit falschem `governedAccessPath` |